In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

print("--- APPROACH 1: THE WINNING LOGISTIC CLASSIFIER ---")
# Data loading & prep (same as File 1)
df = pd.read_csv('combined_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['High'] = df.groupby('Company Name')['High'].ffill()
df['Low'] = df.groupby('Company Name')['Low'].ffill()
df['Volume'] = df['Volume'].fillna(0)
df['Day_Range'] = df['High'] - df['Low']
df['Daily_Return'] = df.groupby('Company Name')['Close'].pct_change() * 100
df['Target_Trend'] = (df['Close'] > df['Open']).astype(int)

features = ['Open', 'High', 'Low', 'Volume', 'Day_Range', 'Daily_Return']
df_clean = df.dropna(subset=features + ['Target_Trend']).copy()
X = df_clean[features]
y = df_clean['Target_Trend']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Training the solver
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)

log_acc = accuracy_score(y_test, log_model.predict(X_test))
print(f"Final Accuracy: {log_acc * 100:.2f}% ★ BEST OVERALL MODEL ★")

cm = confusion_matrix(y_test, log_model.predict(X_test))
print(f"\nSuccessfully caught {cm[1][1]:,} Bullish days and {cm[0][0]:,} Bearish days.")

--- APPROACH 1: THE WINNING LOGISTIC CLASSIFIER ---
Final Accuracy: 87.66% ★ BEST OVERALL MODEL ★

Successfully caught 18,981 Bullish days and 22,753 Bearish days.


In [2]:
print("\n--- APPROACH 2: EXTRACTING MATHEMATICAL COEFFICIENTS ---")
print("Unlike the Random Forest 'Black Box', Logistic Regression gives us exact mathematical weights for every feature:\n")

coef_df = pd.DataFrame({
    'Feature': features,
    'Weight (Impact)': log_model.coef_[0]
})

# Sort by absolute impact 
coef_df['Absolute Impact'] = coef_df['Weight (Impact)'].abs()
coef_df = coef_df.sort_values(by='Absolute Impact', ascending=False).drop(columns=['Absolute Impact'])

for index, row in coef_df.iterrows():
    direction = "BULLISH" if row['Weight (Impact)'] > 0 else "BEARISH"
    print(f"{row['Feature']:<15} : {row['Weight (Impact)']:>8.4f} (Drives trend {direction})")

print("\nExpert Conclusion: By unifying our target to a simple binary trend (Close > Open), this linear probability model trained faster and outperformed the 100-tree ensemble.")


--- APPROACH 2: EXTRACTING MATHEMATICAL COEFFICIENTS ---
Unlike the Random Forest 'Black Box', Logistic Regression gives us exact mathematical weights for every feature:

Open            :  -0.4435 (Drives trend BEARISH)
High            :   0.2266 (Drives trend BULLISH)
Low             :   0.2167 (Drives trend BULLISH)
Daily_Return    :   0.1496 (Drives trend BULLISH)
Day_Range       :   0.0098 (Drives trend BULLISH)
Volume          :  -0.0000 (Drives trend BEARISH)

Expert Conclusion: By unifying our target to a simple binary trend (Close > Open), this linear probability model trained faster and outperformed the 100-tree ensemble.
